In [77]:
import os
from langgraph.graph import StateGraph,START,END
from langchain_groq import ChatGroq
from langchain.messages import SystemMessage,HumanMessage
from typing import Annotated,TypedDict,Optional
from pydantic import BaseModel,Field
from operator import add

Initializing Credentials

In [78]:
OPENAI_MODEL=os.getenv("OPENAI_MODEL")
QWEN_MODEL=os.getenv("QWEN_MODEL")
GROQ_API_KEY=os.getenv("GROQ_API_KEY")

Creating Structured Schemas

In [79]:
class Task(BaseModel):
    """A single section-writing task for a worker node"""
    section_title:str=Field(description="title/heading for this sections")
    order:int=Field(description="What shoudl be the order of this section in final post i-e 1,2,3 etc")
    description:str=Field(description="Detailed Instructions on what to cover in this section")
    word_count:Optional[int]=Field(description="Total/estimated words for this section should be around 250 to 400")
    keywords:Optional[list]=Field(default=None,description="Key terms or concepts that should be covered in this section")
    
    

In [80]:
class Plan(BaseModel):
    """Complete Blog plan generated by orchestrator"""
    title:str=Field(description="Overall blog post title")
    intro_summary:str=Field(description="One-Line summary for the blog's angle/thesis , for context")
    tasks:list[Task]=Field(description="List of task written in this sections",min_length=1,max_length=8)

Blog State

In [81]:
class BlogState(TypedDict):
    topic:str
    plan:Plan
    completed_sections:Annotated[list[str],add]
    final_response:str

Worker State

In [82]:
class WorkerState(TypedDict):
    task: Task                                              # the single task this worker handles
    completed_sections: Annotated[list[str], add]

Setting LLM

In [83]:
llm=ChatGroq(model=OPENAI_MODEL,api_key=GROQ_API_KEY)

Structured Output

In [84]:
structured_llm=llm.with_structured_output(Plan)

NODES

------------------------Orchestrator Node--------------------

In [85]:
def orchestrator_node(state:BlogState):
    topic=state["topic"]
    system_prompt=SystemMessage(content=f"""You are an expert blog content strategist and planner.
                Your job is to break down a blog topic into a clear, logical set of sections 
                that together form a complete, well-structured blog post.""")
    human_message=HumanMessage(content= f"Topic: {topic}\n\n"
                """Create a blog plan. 
                Each section should have a clear title, a description of what it should cover, 
                and its order in the blog. Make sure the flow goes from introduction to conclusion.""")
    
    plan=structured_llm.invoke([system_prompt,human_message])
    print("Orchestrting done...")
    return {"plan":plan}
    

------------------- Fanout Function ------------------------

In [86]:
from langgraph.constants import Send

def assign_workers(state: BlogState):
    """Takes the plan's tasks and dispatches one Send per task, in parallel."""
    return [
        Send("worker", {"task": t})
        for t in state["plan"].tasks
    ]

C:\Users\wasil\AppData\Local\Temp\ipykernel_17628\1315327152.py:1: LangGraphDeprecatedSinceV10: Importing Send from langgraph.constants is deprecated. Please use 'from langgraph.types import Send' instead. Deprecated in LangGraph V1.0 to be removed in V2.0.
  from langgraph.constants import Send


-------------------------------Worker Node------------------------------

In [87]:
def worker_node(state: WorkerState):
    task = state["task"]

    system_prompt = SystemMessage(
        content=(
            "You are an expert blog writer. Write clear, engaging, well-structured "
            "content in Markdown format. Use proper Markdown syntax: ## for the section "
            "heading, **bold** for emphasis, - for bullet points, and short paragraphs."
        )
    )

    human_message = HumanMessage(
        content=(
            f"Write a blog section titled: {task.section_title}\n\n"
            f"Instructions: {task.description}  , Related Tags  : {task.keywords}"
            f"Return only the Markdown content for this section — no extra commentary."
        )
    )

    result = llm.invoke([system_prompt, human_message])
    print("Worker Node done")
    return {"completed_sections": [result.content]}

In [ ]:
def aggregator(state: BlogState):
    # Combine all sections into one markdown string
    combined_markdown = "\n\n".join(state["completed_sections"])
    
    # Add a title at the top
    final_content = f"# {state['plan'].title}\n\n{combined_markdown}"
    
    # Save to a single file
    filename = "blog_output.md"
    with open(filename, "w", encoding="utf-8") as f:
        f.write(final_content)
    
    return {"final_blog": final_content}

In [ ]:
graph=StateGraph(BlogState)

graph.add_node("orchestrator_node",orchestrator_node)
graph.add_node("worker",worker_node)
graph.add_node("aggregator",aggregator)

graph.add_edge(START,"orchestrator_node")
graph.add_conditional_edges("orchestrator_node",assign_workers,"worker")
graph.add_edge("worker","aggregator")

graph.add_edge("aggregator",END)

workflow=graph.compile()

In [89]:
result=workflow.invoke({"topic":"Rust is Gaining Popularity in 2026"})

Orchestrting done...
Worker Node done
Worker Node done
Worker Node done
Worker Node done
Worker Node done
Worker Node done
Worker Node done
Worker Node done


In [90]:
result

{'topic': 'Rust is Gaining Popularity in 2026',
 'plan': Plan(title='Why Rust is Booming in 2026: Trends, Drivers, and What’s Next', intro_summary='An in‑depth look at the factors propelling Rust into the spotlight in 2026, from its unmatched performance and safety guarantees to a thriving ecosystem, expanding industry adoption, and the emerging talent pipeline.', tasks=[Task(section_title='1. Introduction – Rust’s Rise to Prominence', order=1, description='Set the stage by briefly recounting Rust’s journey since its 2010 debut, highlight recent metrics (e.g., Stack Overflow ranking, GitHub stars, Tiobe index) that signal its surge in 2026, and pose the central question: why is Rust suddenly everywhere?', word_count=200, keywords=['Rust language', '2026 popularity', 'programming language trends']), Task(section_title='2. The Core Advantages Fueling Growth', order=2, description='Explain the three technical pillars that continue to attract developers: memory safety without a garbage col